# AI651 Assignment 3 — Q5 (optional): extra seeds for mean ± std
**Student:** Huzaifa | **Roll No:** 25280009 | AI651 Deep Learning for Space, Time and Graphs, Fall 2026, Assignment 3

The primary table (seed 42) is produced by `Q4_Q5_training.ipynb`. Here the *same* protocol is repeated for seeds 43 and 44 (main-table models and the self-path extension) so that Q6 can report mean ± std. The seed changes: weight initialisation, adaptive-graph embedding initialisation, and the training-batch order. Data splits and normalisation statistics are unchanged (they do not depend on the seed).

In [1]:
import os, sys, json
sys.path.insert(0, '../src')
import numpy as np, torch, pandas as pd
from common import set_seed, save_json, load_json, PROC, RES, CKPT
from data import build_data, T_IN, T_OUT, N_NODES
from models import STModel, fixed_adjacency, count_params
from train import CFG, train_model, evaluate, predict, metrics_from
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
A_fixed = fixed_adjacency(np.load(f'{PROC}/sd_adj.npy'), 'row')
SEEDS = [43, 44]
RUNS = [('temporal_only', 'none', False), ('stgnn_fixed', 'fixed', False), ('stgnn_adaptive', 'adaptive', False), ('stgnn_hybrid', 'hybrid', False),
        ('stgnn_fixed_skip', 'fixed', True), ('stgnn_adaptive_skip', 'adaptive', True), ('stgnn_hybrid_skip', 'hybrid', True)]
print('device', DEVICE, '| seeds', SEEDS, '| runs', [r[0] for r in RUNS])

/home/huzaifa/miniconda3/envs/pa1/lib/python3.11/site-packages/torch/cuda/__init__.py:58: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


device cuda | seeds [43, 44] | runs ['temporal_only', 'stgnn_fixed', 'stgnn_adaptive', 'stgnn_hybrid', 'stgnn_fixed_skip', 'stgnn_adaptive_skip', 'stgnn_hybrid_skip']


In [2]:
for seed in SEEDS:
    for name, graph, sp in RUNS:
        out_path = f'{RES}/train_{name}_seed{seed}.json'
        if os.path.exists(out_path):
            print('skip (exists):', out_path); continue
        cfg = dict(CFG); cfg['seed'] = seed
        d_ = build_data(batch_size=cfg['batch_size'], seed=seed, verbose=False)
        set_seed(seed)
        mdl = STModel(graph, N_NODES, d=cfg['hidden_d'], t_in=T_IN, horizon=T_OUT, A_fixed=A_fixed, d_e=cfg['d_e'], top_k=cfg['top_k'], head_hidden=cfg['head_hidden'], self_path=sp)
        info = train_model(mdl, d_, f'{name}', cfg, device=DEVICE, verbose=False)
        P, Y, M = predict(mdl, d_['loaders']['test'], d_['mean'], d_['std'], DEVICE)
        out = dict(name=name, graph=graph, seed=seed, self_path=sp, params=count_params(mdl), device=DEVICE, **{k: info[k] for k in ('epochs_run', 'best_epoch', 'train_time_sec', 'best_val_mae')},
                   history=info['history'], cfg=cfg, test=metrics_from(P, Y, M), val=evaluate(mdl, d_['loaders']['val'], d_['mean'], d_['std'], DEVICE))
        if graph == 'hybrid': out['alpha'] = float(mdl.alpha)
        if graph in ('adaptive', 'hybrid'):
            with torch.no_grad(): np.save(f'{CKPT}/A_adaptive_{name}_seed{seed}.npy', mdl.adp().cpu().numpy().astype(np.float32))
        save_json(out, out_path)
        print(f"seed {seed} {name:22s} epochs {out['epochs_run']:2d} (best {out['best_epoch']:2d}) time {out['train_time_sec']:6.1f}s | val MAE {out['val']['mae']:.3f} | test MAE {out['test']['mae']:.3f} RMSE {out['test']['rmse']:.3f}"
              + (f" alpha {out['alpha']:.3f}" if 'alpha' in out else ''), flush=True)

seed 43 temporal_only          epochs 49 (best 43) time  511.0s | val MAE 52.012 | test MAE 60.872 RMSE 95.639


seed 43 stgnn_fixed            epochs 60 (best 57) time  646.0s | val MAE 77.114 | test MAE 90.822 RMSE 142.998


seed 43 stgnn_adaptive         epochs 58 (best 52) time  727.2s | val MAE 59.503 | test MAE 74.672 RMSE 120.809


seed 43 stgnn_hybrid           epochs 45 (best 39) time  561.2s | val MAE 57.676 | test MAE 71.862 RMSE 115.284 alpha 0.236


seed 43 stgnn_fixed_skip       epochs 42 (best 36) time  455.4s | val MAE 47.051 | test MAE 56.503 RMSE 88.851


seed 43 stgnn_adaptive_skip    epochs 60 (best 54) time  748.1s | val MAE 40.624 | test MAE 50.272 RMSE 79.735


... (4 outputs left out)

seed 44 stgnn_hybrid           epochs 60 (best 54) time  755.3s | val MAE 56.968 | test MAE 72.160 RMSE 117.105 alpha 0.267


seed 44 stgnn_fixed_skip       epochs 52 (best 46) time  561.2s | val MAE 46.830 | test MAE 56.674 RMSE 88.708


seed 44 stgnn_adaptive_skip    epochs 60 (best 59) time  745.3s | val MAE 40.624 | test MAE 50.232 RMSE 79.795


seed 44 stgnn_hybrid_skip      epochs 40 (best 34) time  510.5s | val MAE 41.443 | test MAE 50.806 RMSE 80.413 alpha 0.048


## Summary over seeds 42, 43, 44 (mean ± sample std, test period, original units)

In [3]:
rows = []
for name, graph, sp in RUNS:
    rs = [load_json(f'{RES}/train_{name}_seed{s}.json') for s in (42, 43, 44) if os.path.exists(f'{RES}/train_{name}_seed{s}.json')]
    f = lambda k, sub: np.array([r[sub][k] for r in rs])
    rows.append(dict(model=name, graph=graph, self_path=sp, n_seeds=len(rs), test_MAE=f'{f("mae","test").mean():.2f} ± {f("mae","test").std(ddof=1):.2f}', test_RMSE=f'{f("rmse","test").mean():.2f} ± {f("rmse","test").std(ddof=1):.2f}',
                     val_MAE=f'{f("mae","val").mean():.2f} ± {f("mae","val").std(ddof=1):.2f}', time_s=f'{np.mean([r["train_time_sec"] for r in rs]):.0f} ± {np.std([r["train_time_sec"] for r in rs], ddof=1):.0f}',
                     epochs=[r['epochs_run'] for r in rs]))
S = pd.DataFrame(rows); print(S.to_string(index=False)); S.to_csv(f'{RES}/multiseed_summary.csv', index=False)

              model    graph  self_path  n_seeds     test_MAE     test_RMSE      val_MAE    time_s       epochs
      temporal_only     none      False        3 61.48 ± 0.53  96.49 ± 0.74 52.49 ± 0.45 359 ± 140 [23, 49, 33]
        stgnn_fixed    fixed      False        3 91.11 ± 0.81 143.32 ± 0.48 77.37 ± 0.79  588 ± 96 [45, 60, 60]
     stgnn_adaptive adaptive      False        3 74.82 ± 0.61 121.31 ± 0.80 59.19 ± 0.39  702 ± 58 [52, 58, 60]
       stgnn_hybrid   hybrid      False        3 72.07 ± 0.18 116.12 ± 0.92 57.32 ± 0.35  654 ± 97 [52, 45, 60]
   stgnn_fixed_skip    fixed       True        3 56.50 ± 0.18  88.71 ± 0.14 47.11 ± 0.31 459 ± 100 [34, 42, 52]
stgnn_adaptive_skip adaptive       True        3 50.20 ± 0.09  79.73 ± 0.07 40.79 ± 0.29 667 ± 138 [41, 60, 60]
  stgnn_hybrid_skip   hybrid       True        3 50.62 ± 0.19  80.17 ± 0.21 41.15 ± 0.33 599 ± 147 [41, 60, 40]
